#Chapter 6. 임베딩(Word2Vec~)

#Chapter 6. 임베딩

##6.4 Word2Vec

###6.4.6 모델 실습: Skip-gram

In [ ]:
#예제 6.3 기본 Skip-gram 클래스
from torch import nn

class VanillaSkipgram(nn.Module):
  def __init__(self, vocab_size, embedding_dim):
    super().__init__()
    self.embedding = nn.Embedding(
        num_embeddings=vocab_size,
        embedding_dim=embedding_dim
    )
    self.linear = nn.Linear(
        in_features=embedding_dim,
        out_features=vocab_size
    )

  def forward(self, input_ids):
    embeddings = self.embedding(input_ids)
    output = self.linear(embeddings)
    return output

In [ ]:
!pip install Korpora

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 5.8 MB/s eta 0:00:00


In [ ]:
!pip install konlpy

  Using cached konlpy-0.6.0-py2.py3-none-any.whl.metadata (1.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.9/495.9 kB 44.8 MB/s eta 0:00:00


In [ ]:
#예제 6.4 영화 리뷰 데이터세트 전처리
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus = Korpora.load("nsmc")
corpus = pd.DataFrame(corpus.test)

tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]
print(tokens[:3])


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/



[nsmc] download ratings_train.txt: 14.6MB [00:00, 33.8MB/s]                            
[nsmc] download ratings_test.txt: 4.90MB [00:00, 17.2MB/s]                            


[['굳', 'ㅋ'], ['GDNTOPCLASSINTHECLUB'], ['뭐', '야', '이', '평점', '들', '은', '....', '나쁘진', '않지만', '10', '점', '짜', '리', '는', '더', '더욱', '아니잖아']]


In [ ]:
#예제 6.5 단어 사전 구축
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
  counter = Counter()
  for tokens in corpus:
    counter.update(tokens)
  vocab = special_tokens
  for token, count in counter.most_common(n_vocab):
    vocab.append(token)
  return vocab

vocab = build_vocab(corpus=tokens, n_vocab=5000, special_tokens=["<unk>"])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<unk>', '.', '이', '영화', '의', '..', '가', '에', '...', '을']
5001


In [ ]:
#예제 6.6 Skip-gram의 단어 쌍 추출
def get_word_pairs(tokens, window_size):
  pairs = []
  for sentence in tokens:
    sentence_length = len(sentence)
    for idx, center_word in enumerate(sentence):
      window_start = max(0, idx - window_size)
      window_end = min(sentence_length, idx + window_size + 1)
      context_words = sentence[window_start:idx] + sentence[idx + 1:window_end]
      for context_word in context_words:
        pairs.append((center_word, context_word))
  return pairs

word_pairs = get_word_pairs(tokens, window_size=2)
print(word_pairs[:5])

[('굳', 'ㅋ'), ('ㅋ', '굳'), ('뭐', '야'), ('뭐', '이'), ('야', '뭐')]


In [ ]:
#예제 6.7 인덱스 쌍 변환
def get_index_pairs(word_pairs, token_to_id):
  pairs =[]
  unk_index = token_to_id["<unk>"]
  for word_pair in word_pairs:
    center_word, context_word = word_pair
    center_index = token_to_id.get(center_word, unk_index)
    context_index = token_to_id.get(context_word, unk_index)
    pairs.append([center_index, context_index])
  return pairs

index_pairs = get_index_pairs(word_pairs, token_to_id)
print(index_pairs[:5])

[[595, 100], [100, 595], [77, 176], [77, 2], [176, 77]]


In [ ]:
#예제 6.8 데이터로더 적용
import torch
from torch.utils.data import TensorDataset, DataLoader

index_pairs = torch.tensor(index_pairs)
center_indexes = index_pairs[:,0]
context_indexes = index_pairs[:,1]

dataset = TensorDataset(center_indexes, context_indexes)
dataloader= DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
#예제 6.9 Skip-gram 모델 준비 작업
from torch import optim

device = "cuda" if torch.cuda.is_available() else "cpu"
word2vec = VanillaSkipgram(vocab_size=len(token_to_id), embedding_dim=128).to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(word2vec.parameters(), lr=0.1)

In [ ]:
#예제 6.10 모델 학습
for epoch in range(10):
  cost =0.0
  for input_ids, target_ids in dataloader:
    input_ids = input_ids.to(device)
    target_ids = target_ids.to(device)

    logits = word2vec(input_ids)
    loss = criterion(logits, target_ids)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    cost += loss

  cost = cost/len(dataloader)
  print(f"Epoch: {epoch+1:4d}, Cost : {cost:.3f}")

Epoch:    1, Cost : 6.195
Epoch:    2, Cost : 5.980
Epoch:    3, Cost : 5.931
Epoch:    4, Cost : 5.901
Epoch:    5, Cost : 5.879
Epoch:    6, Cost : 5.861
Epoch:    7, Cost : 5.847
Epoch:    8, Cost : 5.834
Epoch:    9, Cost : 5.822
Epoch:   10, Cost : 5.812


In [ ]:
#예제 6.11 임베딩 값 추출
token_to_embedding = dict()
embedding_matrix = word2vec.embedding.weight.detach().cpu().numpy()

for word, embedding in zip(vocab, embedding_matrix):
  token_to_embedding[word] = embedding

index = 30
token = vocab[30]
token_embedding = token_to_embedding[token]
print(token)
print(token_embedding)

연기
[-1.7997679e-01  3.1625366e-01 -8.0443311e-01  2.4271293e-02
 -8.0043399e-01  3.8046986e-01  6.7200077e-01  1.3751527e+00
  1.0461431e+00 -7.9493940e-01 -2.1675935e-02 -3.4834075e-01
  8.8934648e-01 -4.3758819e-01  4.8279515e-01  1.3433162e+00
 -8.8072673e-05 -1.3931243e+00  1.2152443e-01  1.8324759e+00
  1.6536717e-01  1.2807922e+00  1.9268087e+00 -8.1936073e-01
  4.9351251e-01 -1.1428537e+00 -1.0037934e-01  9.9787641e-01
  5.6524503e-01  4.0995997e-01 -5.2635574e-01 -1.1160196e+00
  9.4713283e-01 -1.7664808e+00 -7.9309070e-01  1.4871994e+00
 -1.0694791e+00  6.0524136e-01  4.0760022e-01 -1.4530272e+00
  8.9918971e-03  1.2084279e+00  3.9935529e-01  1.9492052e+00
  1.1710727e-01 -9.1598064e-01 -3.4232420e-01  2.0048270e-01
 -5.8583575e-01  4.7002187e-01  3.2340997e-01  1.5848203e+00
  5.9444976e-01 -9.6228980e-02  6.7125899e-01 -2.7826977e-01
 -1.9166578e-02 -1.7523574e+00 -1.6548081e-01 -1.9377089e-01
  2.1205618e+00 -1.9133547e-02 -2.0662537e-01 -8.3014053e-01
 -1.0171975e-01 -1.03

In [ ]:
#예제 6.12 단어 임베딩 유사도 계산
import numpy as np
from numpy.linalg import norm

def cosine_similarity(a, b):
  cosine = np.dot(b, a) / (norm(b, axis=1) * norm(a))
  return cosine

def top_n_index(cosine_matrix, n):
  closet_indexes = cosine_matrix.argsort()[::-1]
  top_n = closet_indexes[1: n+1]
  return top_n

cosine_matrix = cosine_similarity(token_embedding, embedding_matrix)
top_n = top_n_index(cosine_matrix, n=5)

print(f"{token}와 가장 유사한 5개 단어")
for index in top_n:
  print(f"{id_to_token[index]} - 유사도 : {cosine_matrix[index]:.4f}")

연기와 가장 유사한 5개 단어
괜찮던데 - 유사도 : 0.3086
자동차 - 유사도 : 0.3043
배급사 - 유사도 : 0.2923
내게 - 유사도 : 0.2919
선물 - 유사도 : 0.2689


###6.4.7 모델 실습: Gensim

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 80.6 MB/s eta 0:00:00


In [ ]:
#예제 6.13 Word2Vec 모델 학습
from gensim.models import Word2Vec

word2vec = Word2Vec(
    sentences = tokens,
    vector_size = 128,
    window =5,
    min_count=1,
    sg = 1,
    epochs = 3,
    max_final_vocab =10000
)

In [ ]:
word2vec.save("/content/drive/MyDrive/Euron9_DL/models/word2vec.model")
word2vec = Word2Vec.load("/content/drive/MyDrive/Euron9_DL/models/word2vec.model")

In [ ]:
#예제 6.14 임베딩 추출 및 유사도 계산
word = "연기"
print(word2vec.wv[word])
print(word2vec.wv.most_similar(word, topn=5))
print(word2vec.wv.similarity(w1=word, w2="연기력"))

[-0.1389229  -0.31980562  0.24839064  0.6249622  -0.1718016   0.26318932
 -0.03740316 -0.04559881 -0.4104547   0.31440642  0.06850731 -0.27481148
 -0.2019803  -0.10889874 -0.35145128  0.15046863 -0.2092245   0.36134678
 -0.27547413  0.24127284  0.6556265   0.3551867  -0.18019214 -0.22619261
 -0.33248502 -0.01571364 -0.40522498  0.23877457  0.14977624 -0.2790515
 -0.34576833  0.0527596   0.3072441  -0.3254767  -0.13218164 -0.47607306
  0.23627935 -0.20932329 -0.19965167 -0.38006523 -0.07975604  0.25568974
 -0.18588178 -0.57215935 -0.22668502  0.24262044 -0.0838187  -0.21660674
  0.16204329  0.05866086  0.77945834  0.3835192   0.07767605  0.2601398
 -0.5270236  -0.14030334  0.1950963   0.36399224 -0.31623995  0.13720414
  0.03308126 -0.0460525   0.03748254  0.18874209 -0.0714258   0.05577822
  0.04968059  0.14396143  0.3163684  -0.07071931 -0.08124378 -0.18096378
 -0.50210243  0.12132427 -0.26821148  0.00818432 -0.15814418 -0.54476833
 -0.13167919  0.07130717  0.04492534 -0.11980546  0.1

##6.5 fastText

###6.5.1 모델 실습

In [ ]:
#예제 6.15 KorNLI 데이터세트 전처리
from Korpora import Korpora

corpus = Korpora.load("kornli")
corpus_texts = corpus.get_all_texts() + corpus.get_all_pairs()
tokens = [sentence.split() for sentence in corpus_texts]

print(tokens[:3])


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : KakaoBrain
    Repository : https://github.com/kakaobrain/KorNLUDatasets
    References :
        - Ham, J., Choe, Y. J., Park, K., Choi, I., & Soh, H. (2020). KorNLI and KorSTS: New Benchmark
           Datasets for Korean Natural Language Understanding. arXiv preprint arXiv:2004.03289.
           (https://arxiv.org/abs/2004.03289)

    This is the dataset repository for our paper
    "KorNLI and KorSTS: New Benchmark Datasets for Korean Natural Language Understanding."
    (https://arxiv.org/abs/2004.03289)
    We introduce KorNLI and KorSTS, which are NLI and STS datasets in Korean.

    # License
    Creative Commons Attribution-ShareAlike license (CC BY-SA 4.0)
    Details in https://creativecommons.org/licenses

[kornli] download multinli.train.ko.tsv: 83.6MB [00:04, 18.9MB/s]                            
[kornli] download snli_1.0_train.ko.tsv: 78.5MB [00:00, 333MB/s]                            
[kornli] download xnli.dev.ko.tsv: 516kB [00:00, 27.4MB/s]
[kornli] download xnli.test.ko.tsv: 1.04MB [00:00, 52.7MB/s]


[['개념적으로', '크림', '스키밍은', '제품과', '지리라는', '두', '가지', '기본', '차원을', '가지고', '있다.'], ['시즌', '중에', '알고', '있는', '거', '알아?', '네', '레벨에서', '다음', '레벨로', '잃어버리는', '거야', '브레이브스가', '모팀을', '떠올리기로', '결정하면', '브레이브스가', '트리플', 'A에서', '한', '남자를', '떠올리기로', '결정하면', '더블', 'A가', '그를', '대신하러', '올라가고', 'A', '한', '명이', '그를', '대신하러', '올라간다.'], ['우리', '번호', '중', '하나가', '당신의', '지시를', '세밀하게', '수행할', '것이다.']]


In [ ]:
#예제 6.16 fastText 모델 실습
from gensim.models import FastText

fastText = FastText(
    sentences = tokens,
    vector_size=128,
    window=5,
    min_count=5,
    sg=1,
    epochs=3,
    min_n=2,
    max_n=6
)

In [ ]:
#예제 6.17 fastText OOV 처리
oov_token = "사랑해요"
oov_vector = fastText.wv[oov_token]

print(oov_token in fastText.wv.index_to_key)
print(fastText.wv.most_similar(oov_vector, topn=5))

False
[('사랑해', 0.9070315957069397), ('사랑한', 0.8564661741256714), ('사랑', 0.8557836413383484), ('사랑해서', 0.8485414385795593), ('사랑해.', 0.8444693684577942)]


##6.6 순환 신경망

###6.6.1 순환 신경망

In [ ]:
#예제 6.18 양방향 다층 신경망
import torch
from torch import nn

input_size = 128
output_size = 256
num_layers = 3
bidirectional = True

model = nn.RNN(
    input_size = input_size,
    hidden_size=output_size,
    num_layers=num_layers,
    nonlinearity='tanh',
    batch_first=True,
    bidirectional=bidirectional
)

batch_size = 4
sequence_len = 6
inputs = torch.randn(batch_size, sequence_len, input_size)
h_0 = torch.rand(num_layers * (int(bidirectional) + 1), batch_size, output_size)

outputs, hidden = model(inputs, h_0)
print(outputs.shape)
print(hidden.shape)

torch.Size([4, 6, 512])
torch.Size([6, 4, 256])


###6.6.2 장단기 메모리

In [ ]:
#예제 6.19 양방향 다층 장단기 메모리
import torch
from torch import nn

input_size = 128
output_size = 256
num_layers = 3
bidirectional = True
proj_size = 64

model =nn.LSTM(
    input_size = input_size,
    hidden_size = output_size,
    num_layers = num_layers,
    batch_first=True,
    bidirectional = bidirectional,
    proj_size = proj_size
)

batch_size = 4
sequence_len = 6

inputs = torch.randn(batch_size, sequence_len, input_size)
h_0 = torch.rand(
    num_layers * (int(bidirectional)+1),
    batch_size,
    proj_size if proj_size > 0 else output_size
)
c_0 = torch.rand(num_layers * (int(bidirectional)+1), batch_size, output_size)

outputs, (h_n, c_n) = model(inputs, (h_0, c_0))

print(outputs.shape)
print(h_n.shape)
print(c_n.shape)


torch.Size([4, 6, 128])
torch.Size([6, 4, 64])
torch.Size([6, 4, 256])


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1124: UserWarning: LSTM with projections is not supported with oneDNN. Using default implementation. (Triggered internally at /pytorch/aten/src/ATen/native/RNN.cpp:1473.)
  result = _VF.lstm(


###6.6.3 모델 실습

In [ ]:
#예제 6.20 문장 분류 모델
from torch import nn
class SentenceClassifier(nn.Module):
  def __init__(
      self,
      n_vocab,
      hidden_dim,
      embedding_dim,
      n_layers,
      dropout=0.5,
      bidirectional=True,
      model_type = "lstm"
  ):
    super().__init__()

    self.embedding = nn.Embedding(
        num_embeddings = n_vocab,
        embedding_dim=embedding_dim,
        padding_idx=0
    )
    if model_type == "rnn":
      self.model = nn.RNN(
          input_size=embedding_dim,
          hidden_size=hidden_dim,
          num_layers=n_layers,
          bidirectional=bidirectional,
          dropout=dropout,
          batch_first=True
      )
    elif model_type == "lstm":
      self.model = nn.LSTM(
          input_size=embedding_dim,
          hidden_size=hidden_dim,
          num_layers=n_layers,
          bidirectional=bidirectional,
          dropout=dropout,
          batch_first=True
      )
    if bidirectional:
      self.classifier = nn.Linear(hidden_dim*2, 1)
    else:
      self.classifier = nn.Linear(hidden_dim, 1)
    self.dropout = nn.Dropout(dropout)

  def forward(self, inputs):
    embeddings = self.embedding(inputs)
    output, _ = self.model(embeddings)
    last_output = output[:, -1, :]
    last_output = self.dropout(last_output)
    logits = self.classifier(last_output)
    return logits

In [ ]:
#예제 6.21 데이터세트 불러오기
import pandas as pd
from Korpora import Korpora
corpus = Korpora.load("nsmc")
corpus_df = pd.DataFrame(corpus.test)

train = corpus_df.sample(frac=0.9, random_state = 42)
test = corpus_df.drop(train.index)

print(train.head(5).to_markdown())
print("Training Data Size : ", len(train))
print("Testing Data Size : ", len(test))


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ra

In [ ]:
#예제 6.22 데이터 토큰화 및 단어 사전 구축
from konlpy.tag import Okt
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
  counter = Counter()
  for tokens in corpus:
    counter.update(tokens)
  vocab = special_tokens
  for token, count in counter.most_common(n_vocab):
    vocab.append(token)
  return vocab

tokenizer = Okt()
train_tokens = [tokenizer.morphs(review) for review in train.text]
test_tokens = [tokenizer.morphs(review) for review in test.text]

vocab = build_vocab(corpus=train_tokens, n_vocab=5000, special_tokens=["<pad>","<unk>"])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<pad>', '<unk>', '.', '이', '영화', '의', '..', '가', '에', '...']
5002


In [ ]:
#예제 6.23 정수 인코딩 및 패딩
import numpy as np

def pad_sequences(sequences, max_length, pad_value):
  result = list()
  for sequence in sequences:
    sequence = sequence[:max_length]
    pad_length = max_length - len(sequence)
    padded_sequence = sequence + [pad_value] * pad_length
    result.append(padded_sequence)
  return np.asarray(result)

unk_id = token_to_id["<unk>"]
train_ids = [
    [token_to_id.get(token, unk_id) for token in review] for review in train_tokens
]
test_ids = [
    [token_to_id.get(token, unk_id) for token in review] for review in test_tokens
]
max_length = 32
pad_id = token_to_id["<pad>"]
train_ids = pad_sequences(train_ids, max_length, pad_id)
test_ids = pad_sequences(test_ids, max_length, pad_id)

print(train_ids[0])
print(test_ids[0])

[ 223 1716   10 4036 2095  193  755    4    2 2330 1031  220   26   13
 4839    1    1    1    2    0    0    0    0    0    0    0    0    0
    0    0    0    0]
[3307    5 1997  456    8    1 1013 3906    5    1    1   13  223   51
    3    1 4684    6    0    0    0    0    0    0    0    0    0    0
    0    0    0    0]


In [ ]:
#예제 6.24 데이터로더 적용
import torch
from torch.utils.data import TensorDataset, DataLoader

train_ids = torch.tensor(train_ids)
test_ids = torch.tensor(test_ids)

train_labels = torch.tensor(train.label.values, dtype=torch.float32)
test_labels = torch.tensor(test.label.values, dtype=torch.float32)

train_dataset = TensorDataset(train_ids, train_labels)
test_dataset = TensorDataset(test_ids, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [ ]:
#예제 6.25 손실 함수와 최적화 함수 정의
from torch import optim

n_vocab = len(token_to_id)
hidden_dim = 64
embedding_dim =128
n_layers = 2

device = "cuda" if torch.cuda.is_available() else "cpu"
classifier = SentenceClassifier(
    n_vocab=n_vocab,
    hidden_dim=hidden_dim,
    embedding_dim=embedding_dim,
    n_layers=n_layers
).to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.RMSprop(classifier.parameters(), lr=0.001)

In [ ]:
#예제 6.26 모델 학습 및 테스트
def train(model, datasets, criterion, optimizer, device, interval):
  model.train()
  losses = list()

  for step, (input_ids, labels) in enumerate(datasets):
    input_ids = input_ids.to(device)
    labels = labels.to(device).unsqueeze(1)

    logits = model(input_ids)
    loss = criterion(logits, labels)
    losses.append(loss.item())

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % interval == 0:
      print(f"Train Loss {step}: {np.mean(losses)}")

def test(model, datasets, criterion, device):
  model.eval()
  losses = list()
  corrects = list()

  for step, (input_ids, labels) in enumerate(datasets):
    input_ids = input_ids.to(device)
    labels = labels.to(device).unsqueeze(1)

    logits = model(input_ids)
    loss = criterion(logits, labels)
    losses.append(loss.item())
    yhat = torch.sigmoid(logits)>.5
    corrects.extend(
        torch.eq(yhat, labels).cpu().tolist()
    )
    print(f"Val Loss: {np.mean(losses)}, Val Accuracy: {np.mean(corrects)}")

epochs = 5
interval = 500
for epoch in range(epochs):
  train(classifier, train_loader, criterion, optimizer, device, interval)
  test(classifier, test_loader, criterion, device)

Train Loss 0: 0.6953163146972656
Train Loss 500: 0.6801551236601885
Train Loss 1000: 0.6591252635170768
Train Loss 1500: 0.6282387841728669
Train Loss 2000: 0.6015958504891884
Train Loss 2500: 0.5824620308088617
Val Loss: 0.2582472562789917, Val Accuracy: 0.875
Val Loss: 0.4571365714073181, Val Accuracy: 0.8125
Val Loss: 0.3978161911169688, Val Accuracy: 0.8541666666666666
Val Loss: 0.4077034518122673, Val Accuracy: 0.828125
Val Loss: 0.41617228984832766, Val Accuracy: 0.8
Val Loss: 0.43490535020828247, Val Accuracy: 0.78125
Val Loss: 0.41859823039599825, Val Accuracy: 0.7946428571428571
Val Loss: 0.42257284745574, Val Accuracy: 0.796875
Val Loss: 0.44401173128022087, Val Accuracy: 0.7986111111111112
Val Loss: 0.43181438744068146, Val Accuracy: 0.8
Val Loss: 0.4555792618881572, Val Accuracy: 0.7670454545454546
Val Loss: 0.4654827838142713, Val Accuracy: 0.765625
Val Loss: 0.4584282751266773, Val Accuracy: 0.7644230769230769
Val Loss: 0.44124720245599747, Val Accuracy: 0.776785714285714

In [ ]:
#예제 6.27 학습된 모델로부터 임베딩 추출
token_to_embedding = dict()
embedding_matrix = classifier.embedding.weight.detach().cpu().numpy()

for word, emb in zip(vocab, embedding_matrix):
  token_to_embedding[word] = emb

token = vocab[1000]
print(token, token_to_embedding[token])

보고싶다 [ 0.00982019  0.11890731 -0.949981   -0.00939121  2.4503071  -0.14429569
  0.3467084  -0.81259567  1.2141061   0.6649243   0.31141526 -1.0865538
  0.26081342  1.799313   -1.2322813  -1.0408458  -1.2769358  -0.6354156
  2.3469706  -2.6444652   1.1047871  -1.0316935   0.6999563   1.8324971
 -1.5236896   1.1057572  -0.4867116  -1.5623014   0.8993682  -1.0343255
  0.17360292 -0.03948972  0.0637541   1.0880594  -1.5942199  -0.08978074
  0.2880031  -0.20777516  0.9901648  -0.62023187 -0.8864098   1.4965599
  0.37431288  1.3748119  -0.01222697 -0.18651298 -0.50548357  0.85589665
 -0.03563595  0.37390965 -1.162774   -1.137786    0.04723332 -2.2540684
 -0.92884314 -0.43387797  0.14983948 -0.93485856  0.60247225  0.5238025
  0.52751374 -0.30092838  0.49787164  1.077676    1.2288836   1.4774432
  0.9863718   0.04841294  0.44018534 -0.6438499  -0.37149674 -0.10471579
  0.8101951   0.83243865 -1.5691401   0.69429404 -2.1981812   0.5534555
 -1.2307049   0.70437104  0.47898847 -0.64602345  1.105

In [ ]:
#예제 6.28 사전 학습된 모델로 임베딩 계층 초기화
from gensim.models import Word2Vec

word2vec = Word2Vec.load("/content/drive/MyDrive/Euron9_DL/models/word2vec.model")
init_embeddings = np.zeros((n_vocab, embedding_dim))

for index, token in id_to_token.items():
  if token not in ["<pad>", "<unk>"]:
    init_embeddings[index] = word2vec.wv[token]

embedding_layer = nn.Embedding.from_pretrained(
    torch.tensor(init_embeddings, dtype=torch.float32))

In [ ]:
#예제 6.29 사전 학습된 임베딩 계층 적용
from torch import nn
class SentenceClassifier(nn.Module):
  def __init__(
      self,
      n_vocab,
      hidden_dim,
      embedding_dim,
      n_layers,
      dropout=0.5,
      bidirectional=True,
      model_type = "lstm",
      pretrained_embedding = None
  ):
    super().__init__()

    self.embedding = nn.Embedding(
        num_embeddings = n_vocab,
        embedding_dim=embedding_dim,
        padding_idx=0
    )
    if pretrained_embedding is not None:
      self.embedding = nn.Embedding.from_pretrained(
          torch.tensor(pretrained_embedding, dtype=torch.float32)
      )
    else:
      self.embedding = nn.Embedding(
          num_embeddings=n_vocab,
          embedding_dim=embedding_dim,
          padding_idx=0
      )
    if bidirectional:
      self.classifier = nn.Linear(hidden_dim*2, 1)
    else:
      self.classifier = nn.Linear(hidden_dim, 1)
    self.dropout = nn.Dropout(dropout)

  def forward(self, inputs):
    embeddings = self.embedding(inputs)
    output, _ = self.model(embeddings)
    last_output = output[:, -1, :]
    last_output = self.dropout(last_output)
    logits = self.classifier(last_output)
    return logits

In [ ]:
#예제 6.30 사전 학습된 임베딩을 사용한 모델 학습
classifier = SentenceClassifier(
    n_vocab=n_vocab, hidden_dim=hidden_dim, embedding_dim=embedding_dim,
    n_layers=n_layers, pretrained_embedding=init_embeddings).to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.RMSprop(classifier.parameters(), lr=0.001)

epochs = 5
interval = 500

for epoch in range(epochs):
  train(classifier, train_loader, criterion, optimizer, device, interval)
  test(classifier, test_loader, criterion, device)

Train Loss 0: 0.6896039843559265
Train Loss 500: 0.6935232366154531
Train Loss 1000: 0.6849988363422714
Train Loss 1500: 0.6738101969473684
Train Loss 2000: 0.6653728453085936
Train Loss 2500: 0.6562828499166931
Val Loss: 0.602745532989502, Val Accuracy: 0.5
Val Loss: 0.628655344247818, Val Accuracy: 0.5625
Val Loss: 0.5852508544921875, Val Accuracy: 0.5833333333333334
Val Loss: 0.5700646936893463, Val Accuracy: 0.640625
Val Loss: 0.5976738691329956, Val Accuracy: 0.6125
Val Loss: 0.5791589468717575, Val Accuracy: 0.6666666666666666
Val Loss: 0.5738842274461474, Val Accuracy: 0.6785714285714286
Val Loss: 0.5977397970855236, Val Accuracy: 0.6640625
Val Loss: 0.6349274747901492, Val Accuracy: 0.6388888888888888
Val Loss: 0.6330106049776077, Val Accuracy: 0.65
Val Loss: 0.6286858347329226, Val Accuracy: 0.6420454545454546
Val Loss: 0.6299683079123497, Val Accuracy: 0.6458333333333334
Val Loss: 0.635664667074497, Val Accuracy: 0.6442307692307693
Val Loss: 0.6218391805887222, Val Accuracy: 

##6.7 합성곱 신경망

###6.7.4 완전 연결 계층

In [ ]:
#예제 6.31 합성곱 모델
import torch
from torch import nn

class CNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Sequential(
        nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )
    self.conv2 = nn.Sequential(
        nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )
    self.fc = nn.Linear(32 * 32 * 32, 10)

  def forward(self, x):
    x = self.conv1(x)
    x = self.conv2(x)
    x = torch.flatten(x)
    x = self.fc(x)
    return x

###6.7.5 모델 실습

In [ ]:
#예제 6.32 합성곱 기반 문장 분류 모델 정의
import torch
from torch import nn

class SentenceClassifier(nn.Module):
  def __init__(self, pretrained_embedding, filter_sizes, max_length, dropout=0.5):
    super().__init__()

    self.embedding = nn.Embedding.from_pretrained(
        torch.tensor(pretrained_embedding, dtype=torch.float32)
    )
    embedding_dim = self.embedding.weight.shape[1]

    conv =[]
    for size in filter_sizes:
      conv.append(
          nn.Sequential(
              nn.Conv1d(
                  in_channels=embedding_dim,
                  out_channels=1,
                  kernel_size=size
              ),
              nn.ReLU(),
              nn.AdaptiveMaxPool1d(1),
              )
          )
    self.conv_filters = nn.ModuleList(conv)

    output_size = len(filter_sizes)
    self.pre_classifier = nn.Linear(output_size, output_size)
    self.dropout = nn.Dropout(dropout)
    self.classifier = nn.Linear(output_size, 1)

  def forward(self, inputs):
    embeddings = self.embedding(inputs)
    embeddings = embeddings.permute(0, 2, 1)

    conv_outputs = [conv(embeddings) for conv in self.conv_filters]
    concat_outputs = torch.cat([conv.squeeze(-1) for conv in conv_outputs], dim=1)

    logits = self.pre_classifier(concat_outputs)
    logits = self.dropout(logits)
    logits = self.classifier(logits)
    return logits

In [ ]:
import pandas as pd
from Korpora import Korpora
corpus = Korpora.load("nsmc")
corpus_df = pd.DataFrame(corpus.test)

train = corpus_df.sample(frac=0.9, random_state = 42)
test = corpus_df.drop(train.index)

print(train.head(5).to_markdown())
print("Training Data Size : ", len(train))
print("Testing Data Size : ", len(test))


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ra

In [ ]:
from konlpy.tag import Okt
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
  counter = Counter()
  for tokens in corpus:
    counter.update(tokens)
  vocab = special_tokens
  for token, count in counter.most_common(n_vocab):
    vocab.append(token)
  return vocab

tokenizer = Okt()
train_tokens = [tokenizer.morphs(review) for review in train.text]
test_tokens = [tokenizer.morphs(review) for review in test.text]

vocab = build_vocab(corpus=train_tokens, n_vocab=5000, special_tokens=["<pad>","<unk>"])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<pad>', '<unk>', '.', '이', '영화', '의', '..', '가', '에', '...']
5002


In [ ]:
import numpy as np

def pad_sequences(sequences, max_length, pad_value):
  result = list()
  for sequence in sequences:
    sequence = sequence[:max_length]
    pad_length = max_length - len(sequence)
    padded_sequence = sequence + [pad_value] * pad_length
    result.append(padded_sequence)
  return np.asarray(result)

unk_id = token_to_id["<unk>"]
train_ids = [
    [token_to_id.get(token, unk_id) for token in review] for review in train_tokens
]
test_ids = [
    [token_to_id.get(token, unk_id) for token in review] for review in test_tokens
]
max_length = 32
pad_id = token_to_id["<pad>"]
train_ids = pad_sequences(train_ids, max_length, pad_id)
test_ids = pad_sequences(test_ids, max_length, pad_id)

print(train_ids[0])
print(test_ids[0])

[ 223 1716   10 4036 2095  193  755    4    2 2330 1031  220   26   13
 4839    1    1    1    2    0    0    0    0    0    0    0    0    0
    0    0    0    0]
[3307    5 1997  456    8    1 1013 3906    5    1    1   13  223   51
    3    1 4684    6    0    0    0    0    0    0    0    0    0    0
    0    0    0    0]


In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

train_ids = torch.tensor(train_ids)
test_ids = torch.tensor(test_ids)

train_labels = torch.tensor(train.label.values, dtype=torch.float32)
test_labels = torch.tensor(test.label.values, dtype=torch.float32)

train_dataset = TensorDataset(train_ids, train_labels)
test_dataset = TensorDataset(test_ids, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [ ]:
#예제 6.33 합성곱 신경망 분류 모델 학습
from torch import optim

n_vocab = len(token_to_id)
hidden_dim = 64
embedding_dim =128
n_layers = 2

device = "cuda" if torch.cuda.is_available() else "cpu"
filter_sizes = [3, 3, 4, 4, 5, 5]
classifier = SentenceClassifier(
    pretrained_embedding=init_embeddings,
    filter_sizes=filter_sizes,
    max_length=max_length
).to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.Adam(classifier.parameters(), lr=0.001)

In [ ]:
def train(model, datasets, criterion, optimizer, device, interval):
  model.train()
  losses = list()

  for step, (input_ids, labels) in enumerate(datasets):
    input_ids = input_ids.to(device)
    labels = labels.to(device).unsqueeze(1)

    logits = model(input_ids)
    loss = criterion(logits, labels)
    losses.append(loss.item())

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % interval == 0:
      print(f"Train Loss {step}: {np.mean(losses)}")

def test(model, datasets, criterion, device):
  model.eval()
  losses = list()
  corrects = list()

  for step, (input_ids, labels) in enumerate(datasets):
    input_ids = input_ids.to(device)
    labels = labels.to(device).unsqueeze(1)

    logits = model(input_ids)
    loss = criterion(logits, labels)
    losses.append(loss.item())
    yhat = torch.sigmoid(logits)>.5
    corrects.extend(
        torch.eq(yhat, labels).cpu().tolist()
    )
    print(f"Val Loss: {np.mean(losses)}, Val Accuracy: {np.mean(corrects)}")

epochs = 5
interval = 500
for epoch in range(epochs):
  train(classifier, train_loader, criterion, optimizer, device, interval)
  test(classifier, test_loader, criterion, device)

Train Loss 0: 0.6863424777984619
Train Loss 500: 0.597717711014186
Train Loss 1000: 0.5781490767633284
Train Loss 1500: 0.5671834697094383
Train Loss 2000: 0.5565438354271522
Train Loss 2500: 0.549853629181262
Val Loss: 0.5278410315513611, Val Accuracy: 0.6875
Val Loss: 0.4808432012796402, Val Accuracy: 0.75
Val Loss: 0.45017876227696735, Val Accuracy: 0.7916666666666666
Val Loss: 0.43942393362522125, Val Accuracy: 0.8125
Val Loss: 0.45507737398147585, Val Accuracy: 0.7875
Val Loss: 0.45073871811230976, Val Accuracy: 0.78125
Val Loss: 0.43806850058691843, Val Accuracy: 0.7946428571428571
Val Loss: 0.44524237141013145, Val Accuracy: 0.7890625
Val Loss: 0.4554503593179915, Val Accuracy: 0.7847222222222222
Val Loss: 0.4543470412492752, Val Accuracy: 0.7875
Val Loss: 0.4664913605559956, Val Accuracy: 0.7784090909090909
Val Loss: 0.4652530774474144, Val Accuracy: 0.7708333333333334
Val Loss: 0.4710869857898125, Val Accuracy: 0.7740384615384616
Val Loss: 0.46157678748880115, Val Accuracy: 0.